In [1]:
%pip install pandas folium


   ---------- ----------------------------- 1/4 [jinja2]
   ---------- ----------------------------- 1/4 [jinja2]
   ---------- ----------------------------- 1/4 [jinja2]
   ---------- ----------------------------- 1/4 [jinja2]
   ---------- ----------------------------- 1/4 [jinja2]
   -------------------- ------------------- 2/4 [branca]
   ------------------------------ --------- 3/4 [folium]
   ------------------------------ --------- 3/4 [folium]
   ------------------------------ --------- 3/4 [folium]
   ------------------------------ --------- 3/4 [folium]
   ------------------------------ --------- 3/4 [folium]
   ------------------------------ --------- 3/4 [folium]
   ------------------------------ --------- 3/4 [folium]
   ------------------------------ --------- 3/4 [folium]
   ------------------------------ --------- 3/4 [folium]
   ------------------------------ --------- 3/4 [folium]
   ---------------------------------------- 4/4 [folium]

Note: you may need to restart


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from pathlib import Path

import pandas as pd
import folium

from folium.plugins import (
    Fullscreen,
    LocateControl,
    MarkerCluster,
    MeasureControl,
    MiniMap,
    MousePosition,
    Search,
)

In [3]:
DATA_URL = (
    "https://data.smartdublin.ie/dataset/"
    "33ec9fe2-4957-4e9a-ab55-c5e917c7a9ab/resource/"
    "2dec86ed-76ed-47a3-ae28-646db5c5b965/download/dublin.csv"
)

try:
    bike_stations = pd.read_csv(DATA_URL)
    print(f"Loaded {len(bike_stations):,} bike stations.")
except Exception as exc:
    raise RuntimeError(
        "The dataset could not be downloaded. "
        "Check the URL or your internet connection."
    ) from exc

bike_stations.head()

Loaded 110 bike stations.


,Number,Name,Address,Latitude,Longitude
0,42,SMITHFIELD NORTH,Smithfield North,53.349562,-6.278198
1,30,PARNELL SQUARE NORTH,Parnell Square North,53.353462,-6.265305
2,54,CLONMEL STREET,Clonmel Street,53.336021,-6.262980
3,108,AVONDALE ROAD,Avondale Road,53.359405,-6.276142
4,56,MOUNT STREET LOWER,Mount Street Lower,53.337960,-6.241530


In [4]:
# Standardise column names by removing unnecessary spaces
bike_stations.columns = bike_stations.columns.str.strip()

required_columns = {"Name", "Address", "Latitude", "Longitude"}
missing_columns = required_columns.difference(bike_stations.columns)

if missing_columns:
    raise ValueError(
        f"Missing required columns: {sorted(missing_columns)}\n"
        f"Available columns: {bike_stations.columns.tolist()}"
    )

# Keep the useful columns
bike_stations = bike_stations[
    ["Number", "Name", "Address", "Latitude", "Longitude"]
].copy()

# Convert coordinates to numeric values
bike_stations["Latitude"] = pd.to_numeric(
    bike_stations["Latitude"],
    errors="coerce"
)

bike_stations["Longitude"] = pd.to_numeric(
    bike_stations["Longitude"],
    errors="coerce"
)

# Remove rows without valid coordinates or names
bike_stations = bike_stations.dropna(
    subset=["Name", "Latitude", "Longitude"]
).drop_duplicates(
    subset=["Latitude", "Longitude", "Name"]
)

# Keep coordinates within a reasonable Dublin-area range
bike_stations = bike_stations[
    bike_stations["Latitude"].between(53.20, 53.50)
    & bike_stations["Longitude"].between(-6.50, -6.00)
].reset_index(drop=True)

print(f"Clean dataset: {len(bike_stations):,} stations.")
bike_stations.head()

Clean dataset: 110 stations.


,Number,Name,Address,Latitude,Longitude
0,42,SMITHFIELD NORTH,Smithfield North,53.349562,-6.278198
1,30,PARNELL SQUARE NORTH,Parnell Square North,53.353462,-6.265305
2,54,CLONMEL STREET,Clonmel Street,53.336021,-6.262980
3,108,AVONDALE ROAD,Avondale Road,53.359405,-6.276142
4,56,MOUNT STREET LOWER,Mount Street Lower,53.337960,-6.241530


In [5]:
centre_latitude = bike_stations["Latitude"].mean()
centre_longitude = bike_stations["Longitude"].mean()

bike_map = folium.Map(
    location=[centre_latitude, centre_longitude],
    zoom_start=13,
    control_scale=True,
    tiles=None,
    prefer_canvas=True
)

# Basemap 1
folium.TileLayer(
    tiles="OpenStreetMap",
    name="OpenStreetMap",
    control=True
).add_to(bike_map)

# Basemap 2
folium.TileLayer(
    tiles="CartoDB positron",
    name="Light Map",
    control=True
).add_to(bike_map)

# Basemap 3
folium.TileLayer(
    tiles="CartoDB dark_matter",
    name="Dark Map",
    control=True
).add_to(bike_map)

In [6]:
station_group = folium.FeatureGroup(
    name="Dublin Bike Stations",
    show=True
)

station_cluster = MarkerCluster(
    name="Clustered Bike Stations",
    control=False,
    show=True,
    options={
        "showCoverageOnHover": False,
        "spiderfyOnMaxZoom": True,
        "disableClusteringAtZoom": 17
    }
)

station_cluster.add_to(station_group)
station_group.add_to(bike_map)

for _, station in bike_stations.iterrows():
    station_name = str(station["Name"])
    station_address = str(station["Address"])
    station_number = station["Number"]

    popup_html = f"""
    <div style="width:240px; font-family:Arial, sans-serif;">
        <h3 style="margin:0 0 10px; color:#0F3D2A;">
            {station_name}
        </h3>

        <p style="margin:4px 0;">
            <strong>Station number:</strong> {station_number}
        </p>

        <p style="margin:4px 0;">
            <strong>Address:</strong> {station_address}
        </p>

        <p style="margin:4px 0;">
            <strong>Latitude:</strong> {station['Latitude']:.6f}
        </p>

        <p style="margin:4px 0 12px;">
            <strong>Longitude:</strong> {station['Longitude']:.6f}
        </p>

        <a
            href="https://www.google.com/maps/search/?api=1&query=
            {station['Latitude']},{station['Longitude']}"
            target="_blank"
            style="
                display:block;
                padding:8px 12px;
                background:#0F3D2A;
                color:white;
                text-align:center;
                text-decoration:none;
                border-radius:6px;
            "
        >
            Open in Google Maps
        </a>
    </div>
    """

    marker = folium.Marker(
        location=[
            station["Latitude"],
            station["Longitude"]
        ],
        tooltip=station_name,
        popup=folium.Popup(
            popup_html,
            max_width=280
        ),
        icon=folium.Icon(
            color="green",
            icon="bicycle",
            prefix="fa"
        ),
        title=station_name
    )

    marker.add_to(station_cluster)

In [7]:
# Search the marker cluster using marker titles
Search(
    layer=station_cluster,
    search_label="title",
    placeholder="Search for a bike station...",
    collapsed=False,
    position="topleft"
).add_to(bike_map)

Fullscreen(
    position="topright",
    title="Open fullscreen",
    title_cancel="Exit fullscreen",
    force_separate_button=True
).add_to(bike_map)

LocateControl(
    position="topright",
    strings={
        "title": "Show my location"
    },
    flyTo=True
).add_to(bike_map)

MeasureControl(
    position="topright",
    primary_length_unit="kilometers",
    secondary_length_unit="meters"
).add_to(bike_map)

MiniMap(
    toggle_display=True,
    position="bottomright"
).add_to(bike_map)

MousePosition(
    position="bottomleft",
    separator=" | ",
    prefix="Coordinates:",
    lat_formatter="function(num) {return L.Util.formatNum(num, 5);}",
    lng_formatter="function(num) {return L.Util.formatNum(num, 5);}"
).add_to(bike_map)

folium.LayerControl(
    position="topright",
    collapsed=False
).add_to(bike_map)

In [8]:
station_bounds = bike_stations[
    ["Latitude", "Longitude"]
].values.tolist()

bike_map.fit_bounds(station_bounds)

In [9]:
bike_map